# Data preparation and audits

[English](00_data_preparation.ipynb) · [简体中文](notebooks/zh-CN/00_data_preparation.ipynb)

Create deduplicated data, fixed splits, and feature caches from the raw gzip file, and run the data-quality audits.


In [1]:
# Locate shared data and modules when running from a subdirectory.
from pathlib import Path
import os
import sys

_notebook_start = Path.cwd().resolve()
_notebook_parents = (_notebook_start, *_notebook_start.parents)
_project_root = next(
    (p for p in _notebook_parents
     if (p / "data/raw/settles.acl16.learning_traces.13m.csv.gz").is_file()),
    None,
)
if _project_root is None:
    _project_root = next(
        (p for p in _notebook_parents if (p / "experiment_pipeline.py").is_file()),
        _notebook_start,
    )
os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from experiment_pipeline import Pipeline
run = Pipeline()
globals().update(run.audit())


Raw file: D:\fujing123\Projects\Data-Analytic\HLR\data\raw\settles.acl16.learning_traces.13m.csv.gz
First 5 rows:


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,1.0,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,2,2
1,0.5,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,2,1
2,1.0,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,1,1
3,0.5,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,2,1
4,1.0,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,1,1


Reused saved duplicate check and cleaned dataset.


,raw_rows,exact_duplicates_removed,clean_rows
0,12854226,81,12854145


Model and EDA input: D:\fujing123\Projects\Data-Analytic\HLR\data\processed\settles.acl16.learning_traces.13m.dedup_v1.csv.gz


,split,start,stop_exclusive,rows
0,Train,0,9254984,9254984
1,Validation,9254984,11568730,2313746
2,Test,11568730,12854145,1285415


Split follows the original CSV row order. Shuffling is confined to Train during fitting.
Reused the feature cache matching the current data and split.
Cache:D:\fujing123\Projects\Data-Analytic\HLR\data\interim\takeaway_hlr_features_v1.dat；539.4 MiB
Base features: ['sqrt(1+history_correct)', 'sqrt(1+history_seen-history_correct)', 'bias']
Train lexemes:18,353; Validation rows with unseen lexemes:1,150; Test rows with unseen lexemes:560
Reused Train-only categorical index cache.
Train categories: 92,392 users, 6 learning languages, 4 UI languages.
Validation unseen users: 590,955
Reused saved basic-validity audit.


,check,rule_hits,clean_rows
0,missing numeric,0,12854145
1,missing categorical/text,0,12854145
2,invalid numeric type,0,12854145
3,numeric range,0,12854145
4,count relationship,0,12854145
5,p_recall/session ratio,0,12854145
6,malformed CSV,0,12854145


Reused saved history-consistency audit.


,check,rows,clean_rows
0,history rollback,47124,12854145
1,cumulative shortfall (includes rollback),101466,12854145
2,same timestamp with differing counts,41,12854145


History anomalies are reported and retained in the cleaned dataset.
